## Packages required

In [1]:
using GridapTopOpt, Gridap, Gridap.TensorValues

## Initial material properties

In [2]:
const E = 30e3        # Young's Modulus
const ν = 0.3         # Poison ratio
const G = E/(2*(1+ν))
const α_mat = 12e-6  #(Thermal expansion coeffiicent)
const κ_mat =  1.0   #(Thermal conductivity)

const Height = 10
const Length = 3*Height

const l = Height/5    # Bending length scale
const N = 0.5         # Micropolar parameter

# const λₘₐₜ = 2*G*ν/(1 -2*ν)  ## For plain strain
# const κₘₐₜ = 2*G*N^2/(1-N^2)
# const μₘₐₜ = G*(1-2*(N^2))/(1-N^2)
# const γₘₐₜ = 4*G*l^2

const λₘₐₜ = E*ν/(1-ν^2)  # Modified λ for plane stress (critical change!)   #2*G*ν/(1 -2*ν)
const κₘₐₜ = 2*G*N^2/(1-N^2)
const μₘₐₜ = G*(1-2*(N^2))/(1-N^2)
const γₘₐₜ = 4*G*l^2

const  T0 = 0.0     # Initial temperature in the domain   
const  TApp = 20.0  # Applied temperature at the boundary
const  fApp = 10    # Mechanical load

10

In [3]:
# FE parameters
order = 1                                                        # Finite element order
dom = (0,Length,0,Height)                                        # Bounding domain
nx,ny = (225,75)
el_size = (nx,ny)
model = CartesianDiscreteModel(dom, el_size)                     # Initialize the model
el_Δ = get_el_Δ(model)
shift = 0.05
f_Γ_D_1(x) = (x[1] ≈ 0.0)                                        # Left side
f_Γ_D_2(x) = (x[1] ≈ Length)                                     # Right side                                               
f_Γ_D_3(x) = (x[2] ≈ 0.0) && ((x[1] < Length/2 - Length/40) || (x[1] > Length/2 + Length/40)) # Bottom face
f_Γ_D_4(x) = (x[2] ≈ Height)  # Top face
f_Γ_D_5(x) = (Length/4 - Length/60 <= x[1] <= Length/4 + Length/60) && (Height/2 - Height/50 <= x[2] <= Height/2 + Height/50) # Left source
f_Γ_D_6(x) = (3*Length/4 - Length/60 <= x[1] <= 3*Length/4 + Length/60) && (Height/2 - Height/50 <= x[2] <= Height/2 + Height/50) # Right source
                                                                                
f_Γ_N(x) = (x[2] ≈ 0.0) && (Length/2 - Length/40 <= x[1] <= Length/2 + Length/40 )            # Γ_N indicator function

update_labels!(1,model,f_Γ_D_1,"Gamma_D_1")                                 

update_labels!(3,model,f_Γ_D_2,"Gamma_D_2")

update_labels!(4,model,f_Γ_D_3,"Gamma_D_3")

update_labels!(5,model,f_Γ_D_4,"Gamma_D_4")

update_labels!(6,model,f_Γ_D_5,"Gamma_D_5")

update_labels!(7,model,f_Γ_D_6,"Gamma_D_6")

update_labels!(2,model,f_Γ_N,"Gamma_N")

In [4]:
# FD parameters
γ = 0.1                                                          # HJ eqn time step coeff
γ_reinit = 0.5                                                   # Reinit. eqn time step coeff
max_steps = 45                                                   # Max steps for advection 
tol = 1e-6                                                       # Reinitialisation tolerance                                   

1.0e-6

In [5]:
I2 = SymTensorValue{2,Float64}(1.0 ,0.0 ,1.0)

E_Matrx = TensorValue(0,1,-1,0)

### New CartesianDiscreteModel

function σ_Tot(ε)
    # Plane stress mechanical stress (λₘₐₜ now uses plane stress version)
    σM = λₘₐₜ*tr(ε)*one(ε) + (2*μₘₐₜ + κₘₐₜ)*ε
    return σM
end

function σ_Temp(T)
    # Plane stress thermal stress: α → α*(1+ν) for plane stress
    ε_th = -α_mat * (1+ν) * T * I2  # Note: (1+ν) factor added!
    σM = λₘₐₜ*tr(ε_th)*one(ε_th) + (2*μₘₐₜ + κₘₐₜ)*ε_th
    return σM
end

function σ_Temp0(T0)
    ε_th = -α_mat * (1+ν) * T0 * I2  # Same adjustment
    σM = λₘₐₜ*tr(ε_th)*one(ε_th) + (2*μₘₐₜ + κₘₐₜ)*ε_th
    return σM
end

# Micropolar and conduction terms remain UNCHANGED
function k_gradTemp(∇)
    return κ_mat * ∇
end

function ε_Skw(∇, θ)
    ∇ᵀ = transpose(∇)
    return 0.5*(∇ᵀ - ∇) - (E_Matrx*θ)
end

function σ_Cmod(ϵ_skew)
    return κₘₐₜ*ϵ_skew
end

function M_mod(∇)
    return γₘₐₜ*∇
end

function Skw(u, θ)
    ∇u = ∇(u)
    ∇ᵀ = transpose(∇u)
    return 0.5*(∇ᵀ - ∇u) - (E_Matrx*θ)
end

Skw (generic function with 1 method)

In [6]:
# Problem parameters
g = VectorValue(0,-fApp)                                            # Load applied
vf = 0.5                                                         # Volume fraction constraint
lsf_func = initial_lsf((12/Length),0.1)                 # Initial level set function
iter_mod = 10                                                    # VTK Output modulo                        
path = "./Result_tol&Max_stepMod/TApp-$TApp,fApp-$fApp/Mesh-$nx,$ny/N_$N/ElasticMicropolar_l_$l/Dim_$Length,$Height/E_$E/"      # Output path
mkpath(path)                                                     # Create path

"./Result_tol&Max_stepMod/TApp-20.0,fApp-10/Mesh-225,75/N_0.5/ElasticMicropolar_l_2.0/Dim_30,10/E_30000.0"

In [7]:
writevtk(model,path*"FixedExampleII")

## Triangulations and measures
Ω = Triangulation(model)
dΩ = Measure(Ω,2*order)
Γ_N = BoundaryTriangulation(model,tags="Gamma_N")
dΓ_N = Measure(Γ_N,2*order)
# Γ_N_Therm = BoundaryTriangulation(model,tags=["Gamma_D_5","Gamma_D_6"])
# dΓ_N_Therm = Measure(Γ_N_Therm,2*order)
vol_D = sum(∫(1)dΩ)

299.9999999999999

In [8]:
## Spaces
reffe = ReferenceFE(lagrangian,VectorValue{2,Float64},order)
reffe_scalar = ReferenceFE(lagrangian,Float64,order)
V = TestFESpace(model,reffe;conformity=:H1,dirichlet_tags=["Gamma_D_1","Gamma_D_2"],dirichlet_masks = [(true, true),(true, true)])
U = TrialFESpace(V,[VectorValue(0.0,0.0),VectorValue(0.0,0.0)])   ## Dispalcement Space

R = TestFESpace(model,reffe_scalar;conformity=:H1,dirichlet_tags=["Gamma_D_1","Gamma_D_2"],dirichlet_masks = [(true),(true)])
S = TrialFESpace(R,[0.0,0.0]) # theta(Micro-Rotation)

Q = TestFESpace(model,reffe_scalar;conformity=:H1,dirichlet_tags=["Gamma_D_1","Gamma_D_2","Gamma_D_3","Gamma_D_4","Gamma_D_5","Gamma_D_6","Gamma_N"],dirichlet_masks = [(true),(true),(true),(true),(true),(true),(true)])
P = TrialFESpace(Q,[T0,T0,TApp,T0,TApp,TApp,TApp]) 

TrialFESpace()

In [9]:
USP = MultiFieldFESpace([U,S,P])
VRQ = MultiFieldFESpace([V,R,Q])

MultiFieldFESpace()

In [10]:
V_φ = TestFESpace(model,reffe_scalar)
V_reg = TestFESpace(model,reffe_scalar,dirichlet_tags=["Gamma_N"])
U_reg = TrialFESpace(V_reg,[0.0])

TrialFESpace()

In [11]:
# Level set and interpolator
φh = interpolate(lsf_func,V_φ)
interp₁ = SmoothErsatzMaterialInterpolation(η = (2)*maximum(get_el_Δ(model)),ϵ = 10^-9)    # η = 2 ×  maximum side length of an element.
I₁,H,DH,ρ = interp₁.I,interp₁.H,interp₁.DH,interp₁.ρ
interp₂ = SmoothErsatzMaterialInterpolation(η = (2)*maximum(get_el_Δ(model)),ϵ = 0.03)    # η = 2 ×  maximum side length of an element.
I₂ = interp₁.I

(::GridapTopOpt.var"#I#298"{GridapTopOpt.var"#H#296"{Vector{Float64}}, Vector{Float64}}) (generic function with 1 method)

In [12]:
writevtk(Ω,path*"initial_lsfFixedII",cellfields=["phi"=>φh,
  "ρ(phi)"=>(ρ ∘ φh),"|nabla(phi)|"=>(norm ∘ ∇(φh))])

(["./Result_tol&Max_stepMod/TApp-20.0,fApp-10/Mesh-225,75/N_0.5/ElasticMicropolar_l_2.0/Dim_30,10/E_30000.0/initial_lsfFixedII.vtu"],)

In [13]:
a((u,θ,T),(w,v,z),φ)  = ∫((I₁ ∘ φ)*( (ε(w) ⊙ (σ_Tot∘(ε(u)))) +  (ε(w) ⊙ (σ_Temp∘(T)))
                                  + ((Skw(w,v)) ⊙ (σ_Cmod∘(ε_Skw∘(∇(u),θ))) ) +  ((∇(v))⋅ (M_mod∘(∇(θ)))) 
                                  - (v*((E_Matrx) ⊙ (σ_Cmod∘(ε_Skw∘(∇(u),θ)))))))dΩ + ∫((I₂ ∘ φ)*((∇(z) ⋅ (k_gradTemp∘(∇(T)))) ))dΩ

lm((w,v,z),φ)= ∫((I₁ ∘ φ)*(ε(w) ⊙ (σ_Temp0(T0))))dΩ + ∫(w ⋅ g)dΓ_N 

lm (generic function with 1 method)

In [14]:
state_map = RepeatingAffineFEStateMap(1, a, [lm],USP,VRQ,V_φ)

RepeatingAffineFEStateMap

In [15]:
evo = FiniteDifferenceEvolver(FirstOrderStencil(2,Float64),model,V_φ;max_steps)
  reinit = FiniteDifferenceReinitialiser(FirstOrderStencil(2,Float64),model,V_φ;tol,γ_reinit)
  ls_evo = LevelSetEvolution(evo,reinit)

LevelSetEvolution{FiniteDifferenceEvolver{1}, FiniteDifferenceReinitialiser{1}}(FiniteDifferenceEvolver{1}(FirstOrderStencil{2, Float64}(), CartesianDiscreteModel(), UnconstrainedFESpace(), [1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  17167, 17168, 17169, 17170, 17171, 17172, 17173, 17174, 17175, 17176], (isperiodic = (false, false), Δ = (0.13333333333333333, 0.13333333333333333), ndof = (226, 76), max_steps = 45, correct_ls = false)), FiniteDifferenceReinitialiser{1}(FirstOrderStencil{2, Float64}(), CartesianDiscreteModel(), UnconstrainedFESpace(), [1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  17167, 17168, 17169, 17170, 17171, 17172, 17173, 17174, 17175, 17176], (isperiodic = (false, false), Δ = (0.13333333333333333, 0.13333333333333333), ndof = (226, 76), max_steps = 2000, tol = 1.0e-6, γ_reinit = 0.5, correct_ls = false)))

In [16]:
function C_plane_stress(λₘₐₜ,μₘₐₜ,κₘₐₜ)
    λ = λₘₐₜ  # Plane stress λ
    μ = μₘₐₜ
    κ = κₘₐₜ
    return SymFourthOrderTensorValue{2}(
        λ+(2μ+κ), 0.0, λ,
        0.0, μ+(κ/2), 0.0,
        λ, 0.0, λ+(2μ+κ)
    )
end

C2 = C_plane_stress(λₘₐₜ,μₘₐₜ,κₘₐₜ)

SymFourthOrderTensorValue{2, Float64, 9}(32967.03296703297, 0.0, 9890.10989010989, 0.0, 11538.461538461537, 0.0, 9890.10989010989, 0.0, 32967.03296703297)

In [17]:
function Cᴴ(r,s,uϕ,φ,dΩ,dΓ_N)
    u_s = uϕ[2s-1]; θ_s = uϕ[2s]; T_s = uϕ[2s+1]
    ∫((I₁ ∘ φ)*(C2 ⊙ (ε(u_s) - α_mat*(T_s-T0)*I2) ⊙ (ε(u_s)- α_mat*(T_s-T0)*I2)))dΩ 
end


J(uϕ,φ) = 1*Cᴴ(1,1,uϕ,φ,dΩ,dΓ_N)
C1(uϕ,φ) = ∫(((ρ ∘ φ) - vf)/vol_D)dΩ;

In [18]:
DC1(q,uϕ,φ) = ∫(-1/vol_D*q*(DH ∘ φ)*(norm ∘ ∇(φ)))dΩ

DC1 (generic function with 1 method)

In [19]:
pcfs = PDEConstrainedFunctionals(J,[C1],state_map,analytic_dJ=nothing,analytic_dC=[DC1])

PDEConstrainedFunctionals:
    num_constraints: 1

In [20]:
α = 4max_steps*γ*maximum(get_el_Δ(model))
a_hilb(p,q) = ∫(α^2*∇(p)⋅∇(q) + p*q)dΩ;
vel_ext = VelocityExtension(a_hilb,U_reg,V_reg)

VelocityExtension

In [21]:
function mine_al_converged(
  m::AugmentedLagrangian;
  L_tol = 1e-6,
  C_tol = 1e-3,
  window = 10
)
  h = m.history
  it = h.niter
  if it <= window
    return false
  end

  # objective stability
  Li = h[:L,it]
  L_prev = h[:L,it-window:it-1]
  A = abs(Li - mean(L_prev)) / max(abs(Li), 1.0) < L_tol

  # constraint satisfaction
  Ci = h[:C,it]
  B = maximum(abs.(Ci)) < C_tol

  return A && B
end

mine_al_converged (generic function with 1 method)

In [22]:
# ## Optimiser
optimiser = AugmentedLagrangian(pcfs,ls_evo,vel_ext,φh;maxiter = 250,γ,verbose=true,constraint_names=[:Vol],converged=mine_al_converged)

AugmentedLagrangian

In [23]:
for (it,uh,φh) in optimiser
    uv,θv,Tv = uh
    data = ["φ"=>φh,"H(φ)"=>(H ∘ φh),"|∇(φ)|"=>(norm ∘ ∇(φh)),"uv"=>uv,"θv"=>θv,"Tv"=>Tv,"ρ(φ)"=>(ρ ∘ φh)]
    iszero(it % iter_mod) && writevtk(Ω,path*"out$it",cellfields= data) 
    write_history(path*"/historymodified$tol,max_steps-$max_steps.txt",optimiser.history)
end

Iteration:   0 | L=1.0002e-01, J=9.8347e-02, Vol=1.1588e-01, γ=1.0000e-01, λ1=0.0000e+00, Λ1=2.4932e-01
Iteration:   1 | L=6.6742e-02, J=6.6503e-02, Vol=4.3829e-02, γ=1.0000e-01, λ1=-1.0928e-02, Λ1=2.4932e-01
Iteration:   2 | L=5.7532e-02, J=5.1089e-02, Vol=1.8770e-01, γ=1.0000e-01, λ1=-5.7726e-02, Λ1=2.4932e-01
Iteration:   3 | L=5.7542e-02, J=4.7896e-02, Vol=1.3038e-01, γ=1.0000e-01, λ1=-9.0232e-02, Λ1=2.4932e-01
Iteration:   4 | L=5.7894e-02, J=5.6360e-02, Vol=1.6619e-02, γ=1.0000e-01, λ1=-9.4375e-02, Λ1=2.4932e-01
Iteration:   5 | L=5.4815e-02, J=4.8439e-02, Vol=6.2418e-02, γ=1.0000e-01, λ1=-1.0994e-01, Λ1=2.7425e-01
Iteration:   6 | L=5.1856e-02, J=5.2051e-02, Vol=-1.7737e-03, γ=1.0000e-01, λ1=-1.0945e-01, Λ1=2.7425e-01
Iteration:   7 | L=4.9639e-02, J=4.9472e-02, Vol=1.5205e-03, γ=1.0000e-01, λ1=-1.0987e-01, Λ1=2.7425e-01
Iteration:   8 | L=4.8852e-02, J=5.1870e-02, Vol=-2.8477e-02, γ=1.0000e-01, λ1=-1.0206e-01, Λ1=2.7425e-01
Iteration:   9 | L=4.9275e-02, J=4.7745e-02, Vol=1.470

In [24]:
it = get_history(optimiser).niter; uh = get_state(pcfs);
uv,θv,Tv = uh
writevtk(Ω,path*"out$it;max_step-$max_steps;tol-$tol",cellfields=["φ"=>φh,"H(φ)"=>(H ∘ φh),"|∇(φ)|"=>(norm ∘ ∇(φh)),"uv"=>uv,"θv"=>θv,"Tv"=>Tv,"ρ(φ)"=>(ρ ∘ φh)])

┌ Warning: detected extension '.0e-6' does not correspond to dataset type.
│ Appending '.vtu' to filename.
└ @ WriteVTK C:\Users\IIT BBSR\.julia\packages\WriteVTK\Be3qm\src\WriteVTK.jl:162


(["./Result_tol&Max_stepMod/TApp-20.0,fApp-10/Mesh-225,75/N_0.5/ElasticMicropolar_l_2.0/Dim_30,10/E_30000.0/out232;max_step-45;tol-1.0e-6.vtu"],)